# Refresh gaming data and test its backup
Choose the source list and inspect the new snapshot and backup destinations. **Run All is read-only by default.** A live run copies the starting database, collects into the new copy, verifies source bytes, and restores its ZIP backup before reporting completion.

Recent mode supports MA monthly PDFs and NY weekly workbooks. Choose history explicitly for other sources. Existing snapshots and approval bindings stay separate. Inspect current evidence in notebooks 94-96.


In [ ]:
base_database_file = "data/staging/refresh_20260912T201854Z/gaming_current.sqlite"
selected_sources = [("MA", "online_sports_betting"), ("NY", "online_sports_betting")]
collection_mode = "recent"  # choose "history" explicitly for other sources
recent_report_limit = 2
run_name = None  # None creates a new UTC timestamped folder name; or supply a new name
backup_file = None  # None uses ~/researchOS-backups/<run_name>.zip; choose another location if desired
run_downloads = False
allow_database_writes = False
# For a new database without prior history, set base_database_file = None.
# selected_sources = None means all registered sources only in explicit history mode.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if (ROOT / "gaming" / "src").is_dir():
    ROOT = ROOT / "gaming"
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from variant_gaming.collect import COLLECTORS
from variant_gaming.refresh import refresh_plan, run_refresh
from variant_gaming.storage import connect_readonly

run_name = run_name or datetime.now(timezone.utc).strftime("refresh_%Y%m%dT%H%M%S%fZ")
run_directory = ROOT / "data/staging" / run_name
base_database_path = ROOT / base_database_file if base_database_file else None
backup_path = Path(backup_file).expanduser() if backup_file else Path.home() / "researchOS-backups" / (run_name + ".zip")
database_path = run_directory / "gaming_current.sqlite"


## 1. Review the source and destination plan
Preview performs only local reads. Both the run directory and backup must be new; the backup must be outside the repository. Recent mode never silently falls back to full history.


In [ ]:
if selected_sources is None and collection_mode != "history":
    raise ValueError("Recent mode supports only an explicit MA/NY source list")
refresh_options = dict(root=ROOT, run_dir=run_directory, backup_path=backup_path,
    base_db=base_database_path, sources=list(COLLECTORS) if selected_sources is None else selected_sources,
    mode=collection_mode, report_limit=recent_report_limit)
preview = refresh_plan(**refresh_options)
plan = pd.DataFrame(preview["sources"])
plan["mode"] = collection_mode
display(plan)
print("Read-only starting database:", preview["base_database"])
print("New snapshot:", preview["database"])
print("New backup:", preview["backup"])


## 2. Collect, validate, and back up
Enable both switches only after reviewing the plan. Failed downloads remain visible; a partial capture is labelled `complete_with_exceptions`. A validation or backup failure stops the run and leaves its failure record for inspection. Matching source hashes do not prove analyst approval or economic comparability.

The wrapper retains source revisions and copies the starting database. It never replaces an approved study. Each ZIP includes raw reports, the captured database, configuration, parser code and manifests, with file hashes and a tested database restore. Keep an off-device copy for protection against losing this computer.


In [ ]:
summary = pd.DataFrame()
refresh_result = None
if run_downloads and allow_database_writes:
    refresh_result = run_refresh(**refresh_options, live=True)
    summary = pd.read_csv(run_directory / "collection_summary.csv")
    print("Run status:", refresh_result["status"])
    print("Backup restore verified:", refresh_result["backup_receipt"]["restore_verified"])
    display(summary)
else:
    print("Collection disabled. Both run_downloads and allow_database_writes must be True.")


In [ ]:
if not summary.empty:
    needs_attention = summary[summary.run_status.ne("completed")
        | ~summary.coverage_status.isin(["ok", "recent_only"])]
    display(needs_attention)
    print("Review reporting-period age and source conflicts in notebooks 94 and 95 before drawing conclusions.")


## 3. Inspect the stored coverage
These queries are read-only. In preview they show the starting database. A recent refresh does not recheck every historical report; the source coverage table retains the earlier full-history assessment for sources not refreshed.


In [ ]:
inspection_path = database_path if database_path.exists() else base_database_path
if inspection_path is not None and inspection_path.exists():
    connection = connect_readonly(inspection_path)
    try:
        coverage = pd.read_sql_query("SELECT * FROM source_coverage", connection)
        stored = pd.read_sql_query("SELECT state_code, vertical, COUNT(*) AS stored_rows, MIN(period_start) AS first_period, MAX(period_end) AS last_period FROM gaming_results GROUP BY state_code, vertical", connection)
    finally:
        connection.close()
    display(plan.merge(coverage, on=["state_code", "vertical"], how="left"))
    display(plan.merge(stored, on=["state_code", "vertical"], how="left"))
else:
    print("No starting database selected. A live collection will create the new snapshot.")
